# **<font color="red">Image Generation (text-to-image)</font>**
## **<font color="blue">Nano Banana `gemini-2.5-flash-image`:</font>**
This model is designed for speed and efficiency, optimized for ***high-volume***, ***low-latency*** tasks.
## **<font color="blue">Nano Banana `gemini-3-pro-image-preview`:</font>** 
This model is designed for ***professional asset production***, utilizing ***advanced reasoning ("Thinking")*** to follow ***complex instructions*** and render high-fidelity text.

In [5]:
from google import genai
from google.genai import types
from PIL import Image

client = genai.Client()

prompt = ("Create a picture of a nano banana dish in a fancy restaurant with a Gemini theme")
response = client.models.generate_content(
    model="gemini-2.5-flash-image",
    contents=[prompt],
)

for part in response.parts:
    if part.text is not None:
        print(part.text)
    elif part.inline_data is not None:
        image = part.as_image()
        image.save("generated_image.png")

Here's your nano banana dish with a Gemini theme! 


In [6]:
"""
Dynamic Brand-Aware Ad Generation Pipeline
"""

import os
import asyncio
from datetime import datetime

from google import genai
from google.genai import types
from google.adk.agents import LlmAgent, SequentialAgent
from google.adk.tools import FunctionTool
from google.adk.tools.google_search_tool import google_search
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner

from config import config


# ============================================================
# CONFIG
# ============================================================

os.environ["GOOGLE_API_KEY"] = config.GOOGLE_API_KEY

APP_NAME = "Dynamic-Ad-Generation"
USER_ID = "user_1"
SESSION_ID = "session_1"

TEXT_MODEL = "gemini-2.0-flash"
IMAGE_MODEL = "gemini-2.5-flash-image"


# ============================================================
# IMAGE TOOL
# ============================================================

async def generate_image_tool(prompt: str) -> dict:
    client = genai.Client()

    response = client.models.generate_content(
        model=IMAGE_MODEL,
        contents=[prompt],
    )

    filename = f"brand_ad_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"

    for part in response.parts:
        if part.inline_data:
            image = part.as_image()
            image.save(filename)

    return {
        "status": "success",
        "filename": filename,
    }


generate_image = FunctionTool(func=generate_image_tool)


# ============================================================
# AGENT 1 — DYNAMIC BRAND RESEARCH AGENT
# ============================================================

research_agent = LlmAgent(
    name="brand_research_agent",
    model=TEXT_MODEL,
    instruction="""
You are an expert brand intelligence and marketing strategist.

GOAL:
Create a premium advertisement for the company mentioned by the user.

STEP 1:
Think about what search queries you need.
Use Google Search tool dynamically.

STEP 2:
From search results extract:
- Official Brand Name
- Tagline (if exists)
- Core Product or Service
- Brand colors (if mentioned)
- Logo presence (describe it if available)
- Target audience
- Emotional positioning

STEP 3:
Generate structured output exactly like this:

BRAND_NAME:
...

TAGLINE:
...

AD_HEADLINE:
...

AD_COPY:
...

VISUAL_DIRECTION:
Describe:
- Where brand name should appear
- Where logo should be placed
- Font style suggestion
- Color scheme
- Atmosphere
- Scene concept
""",
    tools=[google_search],
)


# ============================================================
# AGENT 2 — IMAGE DIRECTOR AGENT
# ============================================================

image_agent = LlmAgent(
    name="image_director_agent",
    model=TEXT_MODEL,
    instruction="""
You are a cinematic AI creative director.

From the previous structured output:

1. Extract BRAND_NAME and VISUAL_DIRECTION.
2. Construct a highly detailed image generation prompt.
3. Ensure the brand name appears clearly in the image.
4. Mention logo placement area (top right, bottom left, etc.).
5. Mention typography style and color palette.

Then call generate_image tool with that prompt.

Finally, tell the user the image filename.
""",
    tools=[generate_image],
)


# ============================================================
# PIPELINE
# ============================================================

pipeline_agent = SequentialAgent(
    name="dynamic_brand_ad_pipeline",
    sub_agents=[research_agent, image_agent],
)


# ============================================================
# RUNNER
# ============================================================

async def main():

    session_service = InMemorySessionService()

    await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=SESSION_ID,
    )

    runner = Runner(
        agent=pipeline_agent,
        app_name=APP_NAME,
        session_service=session_service,
    )

    print("\nRunning Dynamic Brand-Aware Pipeline...\n")

    user_message = types.Content(
        role="user",
        parts=[types.Part(text="Create a premium manufacturing advertisement for Lagozon.")]
    )

    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=SESSION_ID,
        new_message=user_message,
    ):
        if not event.content:
            continue

        for part in event.content.parts:

            if part.text:
                print("\n🧠 TEXT:\n", part.text)

            elif part.function_call:
                print("\n🔧 TOOL CALL →", part.function_call.name)

            elif part.function_response:
                print("\n✅ TOOL RESPONSE →", part.function_response.response)

    print("\n✅ Done.\n")


# For Jupyter
await main()


Running Dynamic Brand-Aware Pipeline...


🧠 TEXT:
 Okay, I can help you create a premium manufacturing advertisement for Lagozon. First, I need to gather some information about the company to inform the ad's content and style.



🧠 TEXT:
 Based on the search results, here's a structured advertisement for Lagozon:

BRAND_NAME: Lagozon Technologies

TAGLINE: Simplifying business complexity and accelerating business results

AD_HEADLINE: **Unlock Manufacturing Excellence with AI-Powered Insights**

AD_COPY: Lagozon Technologies transforms your manufacturing operations with cutting-edge Data and AI solutions. Streamline processes, optimize supply chains, predict equipment issues, and enhance customer service, boosting efficiency and profitability. Leverage our domain-specific AI to forecast better and act faster. With Lagozon, turn real-time data into real-time insights, ensuring a competitive edge in today's dynamic market.

VISUAL_DIRECTION:

Describe:

*   **Brand Name:** Lagozon Techn